# 2022 Yellow Taxi QA Plan
- Load each 2022 monthly Parquet file without down-sampling to avoid bias.
- Inspect schema, data completeness, temporal coverage, and categorical domain constraints.
- Run domain-driven QA checks on trip duration, distance, fares, passenger counts, payment logic, and zone codes.
- Capture aggregated metrics for traceability and summarize detected issues for remediation.

In [1]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from collections import defaultdict
from pathlib import Path
from datetime import timedelta
from IPython.display import display

pd.set_option('display.max_columns', None)

YEAR = 2022
TAXI_TYPE = 'yellow'
INPUT_DIR = Path('../raw')

trip_files = sorted(INPUT_DIR.glob(f"{TAXI_TYPE}_tripdata_{YEAR}-*.parquet"))
if not trip_files:
    raise FileNotFoundError(f"No Parquet files found in {INPUT_DIR} for {YEAR} {TAXI_TYPE} trips.")

zone_lookup = pd.read_csv(INPUT_DIR / 'taxi_zone_lookup.csv')

def iter_monthly(columns=None):
    """Yield each monthly dataframe with optional column projection to keep memory bounded."""
    for path in trip_files:
        df = pd.read_parquet(path, columns=columns)
        df['source_file'] = path.name
        yield df

def get_schema_summary(path):
    """Inspect Parquet schema without loading the full file."""
    parquet_file = pq.ParquetFile(path)
    schema = parquet_file.schema_arrow
    return pd.DataFrame({'name': schema.names, 'type': schema.types})

trip_files

[PosixPath('../raw/yellow_tripdata_2022-01.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-02.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-03.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-04.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-05.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-06.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-07.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-08.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-09.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-10.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-11.parquet'),
 PosixPath('../raw/yellow_tripdata_2022-12.parquet')]

In [2]:
schema_df = get_schema_summary(trip_files[0])
schema_df

,name,type
0,VendorID,int64
1,tpep_pickup_datetime,timestamp[us]
2,tpep_dropoff_datetime,timestamp[us]
3,passenger_count,double
4,trip_distance,double
5,RatecodeID,double
6,store_and_fwd_flag,string
7,PULocationID,int64
8,DOLocationID,int64
9,payment_type,int64


In [3]:
import json
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np
import pandas as pd

# --- Setup Directories ---
OUTPUT_DIR = Path('../processed')
REPORTS_DIR = Path('../reports')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Configuration ---
schema_columns = schema_df['name'].tolist()
candidate_columns = [
    'VendorID','tpep_pickup_datetime','tpep_dropoff_datetime','passenger_count','trip_distance',
    'RatecodeID','store_and_fwd_flag','payment_type','fare_amount','extra','mta_tax','tip_amount',
    'tolls_amount','improvement_surcharge','total_amount','congestion_surcharge','airport_fee',
    'PULocationID','DOLocationID'
]
analysis_columns = [col for col in candidate_columns if col in schema_columns]
fare_component_cols = [col for col in ['fare_amount','extra','mta_tax','tip_amount','tolls_amount','improvement_surcharge','congestion_surcharge','airport_fee'] if col in analysis_columns]

valid_zone_ids = set(zone_lookup['LocationID'])
valid_rate_codes = {1, 2, 3, 4, 5, 6}
valid_payment_types = {1, 2, 3, 4, 5, 6}
valid_store_flags = {'Y', 'N'}

# --- Define QA Filters ---
# Each key is an issue name, and each value is a function that accepts a DataFrame
# and returns a boolean Series (mask) identifying rows with that issue.
FILTERS = {
    'nonpositive_distance': lambda df: df['trip_distance'] <= 0,
    'distance_over_100_miles': lambda df: df['trip_distance'] > 100,
    'invalid_trip_duration': lambda df: df['tpep_dropoff_datetime'] <= df['tpep_pickup_datetime'],
    'duration_over_6_hours': lambda df: df['trip_duration_minutes'] > 360,
    'avg_speed_over_80_mph': lambda df: df['avg_mph'] > 80,
    'missing_avg_mph': lambda df: df['avg_mph'].isna(),
    'passenger_count_negative': lambda df: df['passenger_count'] < 0,
    'passenger_count_over_6': lambda df: df['passenger_count'] > 6,
    'RatecodeID_out_of_spec': lambda df: ~df['RatecodeID'].isin(valid_rate_codes),
    'payment_type_out_of_spec': lambda df: ~df['payment_type'].isin(valid_payment_types),
    'store_and_fwd_flag_out_of_spec': lambda df: ~df['store_and_fwd_flag'].isin(valid_store_flags),
    'pickup_zone_missing_lookup': lambda df: ~df['PULocationID'].isin(valid_zone_ids),
    'dropoff_zone_missing_lookup': lambda df: ~df['DOLocationID'].isin(valid_zone_ids),
    'total_amount_nonpositive': lambda df: df['total_amount'] <= 0,
    'fare_component_mismatch': lambda df: (df[fare_component_cols].sum(axis=1) - df['total_amount']).abs() > 0.01,
    'cash_payment_with_recorded_tip': lambda df: (df['payment_type'] == 2) & (df['tip_amount'] > 0),
    'trip_outside_year': lambda df: (df['tpep_pickup_datetime'].dt.year != YEAR) | (df['tpep_dropoff_datetime'].dt.year != YEAR),
}

# --- Automatically Generate QA Flags from Filters ---
QA_FLAGS = {name: 1 << i for i, name in enumerate(FILTERS.keys())}

# --- Initialize Aggregators ---
monthly_profiles = []
missing_counts = pd.Series(0, index=analysis_columns, dtype='float64')
issue_counts = defaultdict(int)
qa_flag_distribution = Counter()
total_rows = 0

# --- Main Processing Loop ---
print(f"Processing files and saving to: {OUTPUT_DIR}")
for path in trip_files:
    print(f"  - Processing and flagging {path.name}")
    df = pd.read_parquet(path, columns=analysis_columns).copy()
    
    # --- Feature Engineering & Type Conversion ---
    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])
    df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
    df['avg_mph'] = np.where(df['trip_duration_minutes'] > 0, df['trip_distance'] / (df['trip_duration_minutes'] / 60), np.nan)
    df['qa_flags'] = 0

    total_rows += len(df)
    monthly_profiles.append({
        'source_file': path.name, 'rows': len(df),
        'pickup_start': df['tpep_pickup_datetime'].min(), 'pickup_end': df['tpep_pickup_datetime'].max(),
        'dropoff_start': df['tpep_dropoff_datetime'].min(), 'dropoff_end': df['tpep_dropoff_datetime'].max(),
    })
    missing_counts = missing_counts.add(df.isna().sum(), fill_value=0)

    # --- Apply QA Flags Automagically ---
    for issue_name, filter_func in FILTERS.items():
        mask = filter_func(df)
        df.loc[mask, 'qa_flags'] |= QA_FLAGS[issue_name]
    
    # --- Update Summaries from Flags ---
    for issue_name, flag_value in QA_FLAGS.items():
        count = ((df['qa_flags'] & flag_value) > 0).sum()
        if count > 0:
            issue_counts[issue_name] += int(count)
    
    qa_flag_distribution.update(df.loc[df['qa_flags'] > 0, 'qa_flags'])

    # --- Save Processed File ---
    output_path = OUTPUT_DIR / path.name
    df.to_parquet(output_path, index=False)
    print(f"    -> Saved flagged file to {output_path}")

# --- Save Legend and Finalize Summaries ---
legend_path = OUTPUT_DIR / 'qa_flags_legend.json'
with open(legend_path, 'w') as f:
    json.dump(QA_FLAGS, f, indent=4)
print(f"\n-> Saved QA flags legend to: {legend_path}")

# --- Create Summary DataFrames ---
monthly_profiles_df = pd.DataFrame(monthly_profiles).sort_values('source_file')
issue_summary_df = (
    pd.DataFrame([{'issue': issue, 'rows': count, 'pct': count / total_rows} for issue, count in issue_counts.items()])
    .sort_values('rows', ascending=False)
)
qa_flag_distribution_df = (
    pd.DataFrame(qa_flag_distribution.most_common(), columns=['qa_flag', 'rows'])
    .assign(pct=lambda d: d['rows'] / total_rows)
)

# --- Save QA Summary Report ---
summary_report_path = REPORTS_DIR / 'qa_summary.csv'
issue_summary_df.to_csv(summary_report_path, index=False)
print(f"-> Saved QA summary report to: {summary_report_path}")

# --- Caching results for notebook analysis ---
qa_cache = {
    'total_rows': total_rows,
    'monthly_profiles': monthly_profiles_df,
    'issue_summary': issue_summary_df,
    'qa_flags_legend': QA_FLAGS,
    'qa_flag_distribution': qa_flag_distribution_df
}

# --- Display Head of Monthly Profiles ---
qa_cache['monthly_profiles'].head()

Processing files and saving to: ../processed
  - Processing and flagging yellow_tripdata_2022-01.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-01.parquet
  - Processing and flagging yellow_tripdata_2022-02.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-02.parquet
  - Processing and flagging yellow_tripdata_2022-03.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-03.parquet
  - Processing and flagging yellow_tripdata_2022-04.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-04.parquet
  - Processing and flagging yellow_tripdata_2022-05.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-05.parquet
  - Processing and flagging yellow_tripdata_2022-06.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-06.parquet
  - Processing and flagging yellow_tripdata_2022-07.parquet
    -> Saved flagged file to ../processed/yellow_tripdata_2022-07.parquet
  - Processing an

,source_file,rows,pickup_start,pickup_end,dropoff_start,dropoff_end
0,yellow_tripdata_2022-01.parquet,2463931,2008-12-31 22:23:09,2022-05-18 20:41:57,2008-12-31 23:06:56,2022-05-18 20:47:45
1,yellow_tripdata_2022-02.parquet,2979431,2003-01-01 00:10:06,2022-05-24 17:41:50,2003-01-01 12:38:59,2022-05-24 17:43:27
2,yellow_tripdata_2022-03.parquet,3627882,2008-12-31 23:02:37,2022-05-15 21:16:46,2008-12-31 23:44:06,2022-05-16 20:38:57
3,yellow_tripdata_2022-04.parquet,3599920,2008-12-31 23:02:01,2022-05-03 09:36:16,2009-01-01 00:13:52,2022-05-03 09:44:01
4,yellow_tripdata_2022-05.parquet,3588295,2003-01-01 00:06:06,2022-06-01 23:55:30,2003-01-01 00:31:38,2022-06-02 00:03:51


# No need to run this, this is only for viewing data

In [4]:
qa_cache

{'total_rows': 39656098,
 'monthly_profiles':                         source_file     rows        pickup_start  \
 0   yellow_tripdata_2022-01.parquet  2463931 2008-12-31 22:23:09   
 1   yellow_tripdata_2022-02.parquet  2979431 2003-01-01 00:10:06   
 2   yellow_tripdata_2022-03.parquet  3627882 2008-12-31 23:02:37   
 3   yellow_tripdata_2022-04.parquet  3599920 2008-12-31 23:02:01   
 4   yellow_tripdata_2022-05.parquet  3588295 2003-01-01 00:06:06   
 5   yellow_tripdata_2022-06.parquet  3558124 2001-08-23 05:34:45   
 6   yellow_tripdata_2022-07.parquet  3174394 2001-01-01 00:03:14   
 7   yellow_tripdata_2022-08.parquet  3152677 2001-01-01 00:27:45   
 8   yellow_tripdata_2022-09.parquet  3183767 2002-12-31 23:12:49   
 9   yellow_tripdata_2022-10.parquet  3675411 2008-12-31 23:02:01   
 10  yellow_tripdata_2022-11.parquet  3252717 2002-12-31 23:03:33   
 11  yellow_tripdata_2022-12.parquet  3399549 2022-11-30 19:07:12   
 
             pickup_end       dropoff_start         drop

In [5]:
print("QA Flags Legend:")
display(qa_cache['qa_flags_legend'])

print("\nQA Flag Distribution (Top 15):")
display(qa_cache['qa_flag_distribution'].head(15))

QA Flags Legend:


{'nonpositive_distance': 1,
 'distance_over_100_miles': 2,
 'invalid_trip_duration': 4,
 'duration_over_6_hours': 8,
 'avg_speed_over_80_mph': 16,
 'missing_avg_mph': 32,
 'passenger_count_negative': 64,
 'passenger_count_over_6': 128,
 'RatecodeID_out_of_spec': 256,
 'payment_type_out_of_spec': 512,
 'store_and_fwd_flag_out_of_spec': 1024,
 'pickup_zone_missing_lookup': 2048,
 'dropoff_zone_missing_lookup': 4096,
 'total_amount_nonpositive': 8192,
 'fare_component_mismatch': 16384,
 'cash_payment_with_recorded_tip': 32768,
 'trip_outside_year': 65536}


QA Flag Distribution (Top 15):


,qa_flag,rows,pct
0,16384,10451483,0.263553
1,18176,1149018,0.028975
2,1,373082,0.009408
3,8192,215844,0.005443
4,1792,119430,0.003012
5,256,109862,0.002770
6,18177,68050,0.001716
7,16385,49714,0.001254
8,8,45653,0.001151
9,8193,39491,0.000996


In [6]:
qa_cache['total_rows']

39656098

In [7]:
qa_cache['monthly_profiles']

,source_file,rows,pickup_start,pickup_end,dropoff_start,dropoff_end
0,yellow_tripdata_2022-01.parquet,2463931,2008-12-31 22:23:09,2022-05-18 20:41:57,2008-12-31 23:06:56,2022-05-18 20:47:45
1,yellow_tripdata_2022-02.parquet,2979431,2003-01-01 00:10:06,2022-05-24 17:41:50,2003-01-01 12:38:59,2022-05-24 17:43:27
2,yellow_tripdata_2022-03.parquet,3627882,2008-12-31 23:02:37,2022-05-15 21:16:46,2008-12-31 23:44:06,2022-05-16 20:38:57
3,yellow_tripdata_2022-04.parquet,3599920,2008-12-31 23:02:01,2022-05-03 09:36:16,2009-01-01 00:13:52,2022-05-03 09:44:01
4,yellow_tripdata_2022-05.parquet,3588295,2003-01-01 00:06:06,2022-06-01 23:55:30,2003-01-01 00:31:38,2022-06-02 00:03:51
5,yellow_tripdata_2022-06.parquet,3558124,2001-08-23 05:34:45,2023-04-18 14:30:05,2001-08-23 05:57:11,2023-04-18 23:30:39
6,yellow_tripdata_2022-07.parquet,3174394,2001-01-01 00:03:14,2022-08-01 08:36:10,2001-01-01 01:12:47,2022-08-01 23:16:14
7,yellow_tripdata_2022-08.parquet,3152677,2001-01-01 00:27:45,2022-09-01 00:04:50,2001-01-01 00:34:17,2022-09-02 10:02:33
8,yellow_tripdata_2022-09.parquet,3183767,2002-12-31 23:12:49,2022-10-01 00:16:41,2002-12-31 23:52:31,2022-10-04 04:38:02
9,yellow_tripdata_2022-10.parquet,3675411,2008-12-31 23:02:01,2022-11-01 01:27:35,2009-01-01 02:20:59,2022-11-03 17:26:46


In [8]:
qa_cache['missing_summary'].head(15)

KeyError: 'missing_summary'

In [ ]:
qa_cache['missing_summary'].head(10).to_string(index=False)

'              column  missing_rows  missing_pct\n         airport_fee       1368303     0.034504\n          RatecodeID       1368303     0.034504\n     passenger_count       1368303     0.034504\n  store_and_fwd_flag       1368303     0.034504\ncongestion_surcharge       1368303     0.034504\n             avg_mph         32035     0.000808\n            VendorID             0     0.000000\n        PULocationID             0     0.000000\n        DOLocationID             0     0.000000\n         fare_amount             0     0.000000'

In [ ]:
qa_cache['issue_summary'].sort_values('rows', ascending=False).head(15)

,issue,rows,pct
10,fare_component_mismatch,11746888,0.296219
6,RatecodeID_out_of_spec,1502097,0.037878
7,payment_type_out_of_spec,1368303,0.034504
8,store_and_fwd_flag_out_of_spec,1368303,0.034504
0,nonpositive_distance,574059,0.014476
9,total_amount_nonpositive,262015,0.006607
4,avg_speed_over_80_mph,49250,0.001242
3,duration_over_6_hours,46979,0.001185
2,nonpositive_duration_minutes,32035,0.000808
11,cash_payment_with_recorded_tip,1753,0.000044


In [ ]:
from collections import Counter
fare_diff_counter = Counter()
congestion_missing_trigger = 0
for path in trip_files:
    cols = list(set(fare_component_cols + ['total_amount', 'congestion_surcharge']))
    df = pd.read_parquet(path, columns=cols)
    comp_sum = df[fare_component_cols].sum(axis=1)
    diff = (df['total_amount'] - comp_sum).round(2)
    mismatch_mask = diff.abs() > 0.01
    fare_diff_counter.update(diff[mismatch_mask])
    congestion_missing_trigger += df.loc[mismatch_mask, 'congestion_surcharge'].isna().sum()

counter_df = (
    pd.DataFrame(fare_diff_counter.most_common(10), columns=['difference','rows'])
    if fare_diff_counter else pd.DataFrame(columns=['difference','rows'])
)

counter_df, congestion_missing_trigger

(   difference     rows
 0       -2.50  9709130
 1        2.50  1256612
 2       -3.75   367008
 3       -1.25   323815
 4        1.95    41596
 5        4.50    24672
 6       -2.80     7285
 7        5.00     7000
 8        3.90     5629
 9        2.00     1482,
 np.int64(1219662))

In [ ]:
qa_cache['passenger_distribution']

,rows,pct
passenger_count,,
0.0,763344,0.019249
1.0,28261419,0.712663
2.0,5864480,0.147883
3.0,1543572,0.038924
4.0,709227,0.017884
5.0,686255,0.017305
6.0,459086,0.011577
7.0,224,0.000006
8.0,140,0.000004


In [ ]:
qa_cache['payment_distribution']

,rows,pct
payment_type,,
0,1368303.0,3.450423e-02
1,30085763.0,7.586668e-01
2,7763339.0,1.957666e-01
3,194323.0,4.900205e-03
4,244364.0,6.162079e-03
5,6.0,1.513008e-07


In [ ]:
qa_cache['ratecode_distribution']

,rows,pct
RatecodeID,,
1.0,36178390,0.912303
2.0,1501318,0.037858
3.0,109154,0.002753
4.0,51722,0.001304
5.0,313064,0.007894
6.0,353,0.000009
99.0,133794,0.003374
NaN,1368303,0.034504


In [ ]:
qa_cache['tip_summary']

,sum,count,mean_tip
payment_type,,,
0,1.829990e+08,1368303.0,133.741555
1,1.038881e+08,30085763.0,3.453067
2,8.360850e+03,7763339.0,0.001077
3,3.618200e+02,194323.0,0.001862
4,1.239523e+04,244364.0,0.050724
5,0.000000e+00,6.0,0.000000


In [ ]:
list(qa_cache['issue_examples'].keys())

['nonpositive_distance',
 'distance_over_100_miles',
 'nonpositive_duration_minutes',
 'total_amount_nonpositive',
 'fare_component_mismatch']

In [ ]:
for issue, sample in qa_cache['issue_examples'].items():
    display(issue)
    display(sample)

'nonpositive_distance'

,source_file,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type,RatecodeID,PULocationID,DOLocationID
21,yellow_tripdata_2022-01.parquet,2022-01-01 00:06:10,2022-01-01 00:08:58,1.0,0.0,2.800000,2.5,0.00,6.30,4,1.0,161,161
141,yellow_tripdata_2022-01.parquet,2022-01-01 00:41:54,2022-01-01 00:42:17,1.0,0.0,0.383333,2.5,0.00,6.30,2,1.0,249,249
144,yellow_tripdata_2022-01.parquet,2022-01-01 00:23:57,2022-01-01 00:24:49,0.0,0.0,0.866667,2.5,0.00,6.30,2,1.0,263,263
245,yellow_tripdata_2022-01.parquet,2022-01-01 00:49:57,2022-01-01 00:50:17,1.0,0.0,0.333333,2.5,0.00,6.30,2,1.0,79,79
362,yellow_tripdata_2022-01.parquet,2022-01-01 00:39:28,2022-01-01 00:39:47,1.0,0.0,0.316667,10.0,2.56,15.36,1,5.0,141,140


'distance_over_100_miles'

,source_file,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type,RatecodeID,PULocationID,DOLocationID
23729,yellow_tripdata_2022-01.parquet,2022-01-01 11:42:44,2022-01-01 12:00:40,3.0,179.6,17.933333,17.5,4.15,24.95,1,1.0,263,144
29219,yellow_tripdata_2022-01.parquet,2022-01-01 13:30:05,2022-01-01 13:53:48,2.0,214.1,23.716667,22.0,5.05,30.35,1,1.0,144,7
71368,yellow_tripdata_2022-01.parquet,2022-01-02 10:59:53,2022-01-02 11:19:12,1.0,172.8,19.316667,18.5,0.00,21.80,1,1.0,238,137
85751,yellow_tripdata_2022-01.parquet,2022-01-02 14:35:42,2022-01-02 14:52:48,1.0,120.2,17.100000,15.0,5.00,23.30,1,1.0,230,24
99158,yellow_tripdata_2022-01.parquet,2022-01-02 17:19:52,2022-01-02 17:40:12,2.0,121.1,20.333333,15.5,4.70,23.50,1,1.0,162,45


'nonpositive_duration_minutes'

,source_file,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type,RatecodeID,PULocationID,DOLocationID
1760,yellow_tripdata_2022-01.parquet,2022-01-01 00:36:35,2022-01-01 00:36:35,1.0,0.0,0.0,115.0,0.0,117.55,2,1.0,132,264
3156,yellow_tripdata_2022-01.parquet,2022-01-01 00:04:59,2022-01-01 00:04:59,1.0,0.0,0.0,2.5,0.0,3.80,2,1.0,264,264
4726,yellow_tripdata_2022-01.parquet,2022-01-01 01:59:36,2022-01-01 01:59:36,1.0,0.0,0.0,2.5,0.0,6.30,2,1.0,237,264
5612,yellow_tripdata_2022-01.parquet,2022-01-01 01:38:42,2022-01-01 01:38:42,2.0,0.0,0.0,6.0,0.0,9.80,2,1.0,141,264
5613,yellow_tripdata_2022-01.parquet,2022-01-01 01:42:30,2022-01-01 01:42:30,2.0,1.3,0.0,6.0,1.2,11.00,1,1.0,263,263


'total_amount_nonpositive'

,source_file,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type,RatecodeID,PULocationID,DOLocationID
154,yellow_tripdata_2022-01.parquet,2022-01-01 00:26:37,2022-01-01 00:39:35,1.0,7.38,12.966667,-21.0,0.0,-23.55,4,1.0,132,265
521,yellow_tripdata_2022-01.parquet,2022-01-01 00:27:18,2022-01-01 00:40:50,1.0,5.14,13.533333,-17.0,0.0,-20.80,4,1.0,152,48
523,yellow_tripdata_2022-01.parquet,2022-01-01 00:59:33,2022-01-01 01:14:09,4.0,5.37,14.600000,-75.0,0.0,-77.80,2,5.0,50,265
580,yellow_tripdata_2022-01.parquet,2022-01-01 00:16:58,2022-01-01 00:19:41,1.0,0.46,2.716667,-4.0,0.0,-7.80,2,1.0,90,234
714,yellow_tripdata_2022-01.parquet,2022-01-01 00:29:12,2022-01-01 00:40:29,1.0,3.13,11.283333,-12.0,0.0,-15.80,4,1.0,13,246


'fare_component_mismatch'

,source_file,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type,RatecodeID,PULocationID,DOLocationID,fare_component_sum,difference
0,yellow_tripdata_2022-01.parquet,2022-01-01 00:35:40,2022-01-01 00:53:29,2.0,3.8,17.816667,14.5,3.65,21.95,1,1.0,142,236,24.45,-2.5
5,yellow_tripdata_2022-01.parquet,2022-01-01 00:40:15,2022-01-01 01:09:48,1.0,10.3,29.550000,33.0,13.00,56.35,1,1.0,138,161,58.85,-2.5
16,yellow_tripdata_2022-01.parquet,2022-01-01 00:33:52,2022-01-01 00:47:28,3.0,4.2,13.600000,14.0,3.45,20.75,1,1.0,148,141,23.25,-2.5
17,yellow_tripdata_2022-01.parquet,2022-01-01 00:53:54,2022-01-01 01:05:20,2.0,2.2,11.433333,9.5,2.55,15.35,1,1.0,237,107,17.85,-2.5
19,yellow_tripdata_2022-01.parquet,2022-01-01 00:35:50,2022-01-01 00:48:33,2.0,3.9,12.716667,13.0,3.35,20.15,1,1.0,107,263,22.65,-2.5


# payment_type = 0


In [ ]:
from collections import Counter

payment0_cols = list(dict.fromkeys(analysis_columns + ['VendorID']))

payment0_total_rows = 0

payment0_null_counts = pd.Series(0, index=analysis_columns, dtype='float64')

payment0_file_counts = Counter()

payment0_month_counts = Counter()

payment0_vendor_counts = Counter()

payment0_ratecode_counts = Counter()

payment0_passenger_counts = Counter()

payment0_tip_positive = 0

payment0_tip_over_100 = 0

payment0_congestion_nulls = 0

payment0_fare_diff_counter = Counter()

payment0_issue_flags = defaultdict(int)

payment0_distance_values = []

payment0_duration_values = []

payment0_tip_values = []

payment0_total_values = []

payment0_example = None



for path in trip_files:

    df = pd.read_parquet(path, columns=payment0_cols).copy()

    df['source_file'] = path.name

    df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])

    df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

    df['trip_duration_minutes'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60

    df['avg_mph'] = np.where(df['trip_duration_minutes'] > 0, df['trip_distance'] / (df['trip_duration_minutes'] / 60), np.nan)

    subset = df[df['payment_type'] == 0].copy()

    if subset.empty:

        continue



    payment0_total_rows += len(subset)

    payment0_file_counts.update(subset['source_file'])

    payment0_month_counts.update(subset['tpep_pickup_datetime'].dt.to_period('M').astype(str))

    payment0_vendor_counts.update(subset['VendorID'].fillna(-1).astype('Int64'))

    payment0_ratecode_counts.update(subset['RatecodeID'].fillna(-1).astype('Int64'))

    payment0_passenger_counts.update(subset['passenger_count'].fillna(-1))

    payment0_null_counts = payment0_null_counts.add(subset.isna().sum(), fill_value=0)



    tip_positive_mask = subset['tip_amount'] > 0

    payment0_tip_positive += int(tip_positive_mask.sum())

    tip_over_100_mask = subset['tip_amount'] >= 100

    payment0_tip_over_100 += int(tip_over_100_mask.sum())

    payment0_congestion_nulls += subset['congestion_surcharge'].isna().sum()



    distance_nonpositive = subset['trip_distance'] <= 0

    duration_nonpositive = subset['trip_duration_minutes'] <= 0

    total_nonpositive = subset['total_amount'] <= 0

    avg_speed_extreme = subset['avg_mph'] > 80

    passenger_over_6 = subset['passenger_count'] > 6



    payment0_issue_flags['nonpositive_distance'] += int(distance_nonpositive.sum())

    payment0_issue_flags['nonpositive_duration'] += int(duration_nonpositive.sum())

    payment0_issue_flags['total_amount_nonpositive'] += int(total_nonpositive.sum())

    payment0_issue_flags['avg_speed_over_80'] += int(avg_speed_extreme.sum())

    payment0_issue_flags['passenger_count_over_6'] += int(passenger_over_6.sum())



    fare_components = subset[fare_component_cols].sum(axis=1)

    fare_diff = (subset['total_amount'] - fare_components).round(2)

    mismatch_mask = fare_diff.abs() > 0.01

    payment0_issue_flags['fare_component_mismatch'] += int(mismatch_mask.sum())

    payment0_fare_diff_counter.update(fare_diff[mismatch_mask])



    payment0_distance_values.extend(subset['trip_distance'].dropna().tolist())

    payment0_duration_values.extend(subset['trip_duration_minutes'].dropna().tolist())

    payment0_tip_values.extend(subset['tip_amount'].dropna().tolist())

    payment0_total_values.extend(subset['total_amount'].dropna().tolist())



    if payment0_example is None:

        display_cols = [col for col in ['source_file','tpep_pickup_datetime','tpep_dropoff_datetime','passenger_count','trip_distance','trip_duration_minutes','fare_amount','tip_amount','total_amount','payment_type','RatecodeID','VendorID','PULocationID','DOLocationID'] if col in subset.columns]

        payment0_example = subset.loc[:, display_cols].head(5)



payment0_null_summary = (

    pd.DataFrame({

        'column': payment0_null_counts.index,

        'missing_rows': payment0_null_counts.astype(int)

    })

    .assign(missing_pct=lambda d: d['missing_rows'] / payment0_total_rows)

    .sort_values('missing_pct', ascending=False)

)



payment0_vendor_summary = (

    pd.Series(payment0_vendor_counts, name='rows')

    .to_frame()

    .assign(pct=lambda d: d['rows'] / payment0_total_rows)

    .sort_values('rows', ascending=False)

)



payment0_ratecode_summary = (

    pd.Series(payment0_ratecode_counts, name='rows')

    .to_frame()

    .assign(pct=lambda d: d['rows'] / payment0_total_rows)

    .sort_values('rows', ascending=False)

)



payment0_passenger_summary = (

    pd.Series(payment0_passenger_counts, name='rows')

    .to_frame()

    .assign(pct=lambda d: d['rows'] / payment0_total_rows)

    .sort_index()

)



payment0_file_summary = (

    pd.Series(payment0_file_counts, name='rows')

    .to_frame()

    .assign(pct=lambda d: d['rows'] / payment0_total_rows)

    .sort_values('rows', ascending=False)

)



payment0_month_summary = (

    pd.Series(payment0_month_counts, name='rows')

    .to_frame()

    .assign(pct=lambda d: d['rows'] / payment0_total_rows)

    .sort_index()

)



payment0_issue_summary = (

    pd.DataFrame([

        {'issue': issue, 'rows': count, 'pct_within_payment0': count / payment0_total_rows}

        for issue, count in payment0_issue_flags.items()

    ]).sort_values('rows', ascending=False)

)



payment0_fare_diff_summary = (

    pd.DataFrame(payment0_fare_diff_counter.most_common(10), columns=['difference','rows'])

    if payment0_fare_diff_counter else pd.DataFrame(columns=['difference','rows'])

)



summary_stats = pd.Series({

    'rows': payment0_total_rows,

    'pct_of_all_rows': payment0_total_rows / qa_cache['total_rows'],

    'mean_trip_distance': float(np.mean(payment0_distance_values)) if payment0_distance_values else np.nan,

    'median_trip_distance': float(np.median(payment0_distance_values)) if payment0_distance_values else np.nan,

    'mean_trip_duration_min': float(np.mean(payment0_duration_values)) if payment0_duration_values else np.nan,

    'median_trip_duration_min': float(np.median(payment0_duration_values)) if payment0_duration_values else np.nan,

    'mean_tip_amount': float(np.mean(payment0_tip_values)) if payment0_tip_values else np.nan,

    'median_tip_amount': float(np.median(payment0_tip_values)) if payment0_tip_values else np.nan,

    'mean_total_amount': float(np.mean(payment0_total_values)) if payment0_total_values else np.nan,

    'median_total_amount': float(np.median(payment0_total_values)) if payment0_total_values else np.nan,

    'share_with_positive_tip': payment0_tip_positive / payment0_total_rows if payment0_total_rows else np.nan,

    'share_with_tip_>=100': payment0_tip_over_100 / payment0_total_rows if payment0_total_rows else np.nan,

    'share_congestion_null': payment0_congestion_nulls / payment0_total_rows if payment0_total_rows else np.nan

}).to_frame('value')



payment0_cache = {

    'summary_stats': summary_stats,

    'null_summary': payment0_null_summary,

    'vendor_summary': payment0_vendor_summary,

    'ratecode_summary': payment0_ratecode_summary,

    'passenger_summary': payment0_passenger_summary,

    'file_summary': payment0_file_summary,

    'month_summary': payment0_month_summary,

    'issue_summary': payment0_issue_summary,

    'fare_diff_summary': payment0_fare_diff_summary,

    'example_rows': payment0_example

}



summary_stats

,value
rows,1.368303e+06
pct_of_all_rows,3.450423e-02
mean_trip_distance,7.437528e+01
median_trip_distance,3.000000e+00
mean_trip_duration_min,2.024482e+01
median_trip_duration_min,1.688333e+01
mean_tip_amount,1.337416e+02
median_tip_amount,3.050000e+00
mean_total_amount,2.859778e+01
median_total_amount,2.300000e+01


In [ ]:
payment0_cache['null_summary'].head(10)

,column,missing_rows,missing_pct
airport_fee,airport_fee,1368303,1.000000
RatecodeID,RatecodeID,1368303,1.000000
passenger_count,passenger_count,1368303,1.000000
store_and_fwd_flag,store_and_fwd_flag,1368303,1.000000
congestion_surcharge,congestion_surcharge,1368303,1.000000
avg_mph,avg_mph,14014,0.010242
VendorID,VendorID,0,0.000000
PULocationID,PULocationID,0,0.000000
DOLocationID,DOLocationID,0,0.000000
fare_amount,fare_amount,0,0.000000


In [ ]:
payment0_cache['vendor_summary'].head(5), payment0_cache['ratecode_summary'].head(5)

(      rows       pct
 2  1020109  0.745529
 1   288450  0.210809
 6    59601  0.043558
 5      143  0.000105,
        rows  pct
 -1  1368303  1.0)

In [ ]:
payment0_cache['passenger_summary'].head(10)

,rows,pct
-1.0,1368303,1.0


In [ ]:
payment0_cache['issue_summary'], payment0_cache['fare_diff_summary']

(                      issue     rows  pct_within_payment0
 5   fare_component_mismatch  1219662             0.891368
 0      nonpositive_distance    71150             0.051999
 3         avg_speed_over_80    14506             0.010601
 1      nonpositive_duration    14014             0.010242
 2  total_amount_nonpositive      736             0.000538
 4    passenger_count_over_6        0             0.000000,
    difference     rows
 0        2.50  1192708
 1        4.50    24672
 2        2.00     1482
 3        0.75      650
 4        5.50       33
 5       17.50       32
 6       15.00       23
 7        3.00       18
 8       -2.50       16
 9        2.75        9)

In [ ]:
payment0_cache['file_summary'].head(12), payment0_cache['month_summary']

(                                   rows       pct
 yellow_tripdata_2022-10.parquet  133021  0.097216
 yellow_tripdata_2022-06.parquet  132448  0.096797
 yellow_tripdata_2022-05.parquet  129524  0.094660
 yellow_tripdata_2022-12.parquet  126463  0.092423
 yellow_tripdata_2022-11.parquet  121958  0.089131
 yellow_tripdata_2022-04.parquet  118874  0.086877
 yellow_tripdata_2022-03.parquet  117814  0.086102
 yellow_tripdata_2022-09.parquet  116592  0.085209
 yellow_tripdata_2022-07.parquet  105194  0.076879
 yellow_tripdata_2022-02.parquet  101738  0.074353
 yellow_tripdata_2022-08.parquet   93174  0.068095
 yellow_tripdata_2022-01.parquet   71503  0.052257,
            rows       pct
 2022-01   71503  0.052257
 2022-02  101738  0.074353
 2022-03  117814  0.086102
 2022-04  118874  0.086877
 2022-05  129524  0.094660
 2022-06  132448  0.096797
 2022-07  105194  0.076879
 2022-08   93174  0.068095
 2022-09  116592  0.085209
 2022-10  133021  0.097216
 2022-11  121958  0.089131
 2022-12  12

In [ ]:
payment0_cache['example_rows']

,source_file,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,trip_duration_minutes,fare_amount,tip_amount,total_amount,payment_type,RatecodeID,VendorID,PULocationID,DOLocationID
2392428,yellow_tripdata_2022-01.parquet,2022-01-01 00:50:00,2022-01-01 00:54:00,NaN,1.00,4.0,13.20,1.75,18.25,0,NaN,2,68,246
2392429,yellow_tripdata_2022-01.parquet,2022-01-01 00:49:24,2022-01-01 01:27:36,NaN,13.31,38.2,44.87,10.05,55.72,0,NaN,2,257,223
2392430,yellow_tripdata_2022-01.parquet,2022-01-01 00:42:00,2022-01-01 00:56:00,NaN,2.87,14.0,13.23,3.51,20.04,0,NaN,2,143,236
2392431,yellow_tripdata_2022-01.parquet,2022-01-01 00:40:00,2022-01-01 00:55:00,NaN,3.24,15.0,14.19,3.72,21.21,0,NaN,2,143,262
2392432,yellow_tripdata_2022-01.parquet,2022-01-01 00:40:00,2022-01-01 00:52:00,NaN,2.19,12.0,13.20,5.25,21.75,0,NaN,2,239,166


In [ ]:
{

    'tip_positive_rows': payment0_tip_positive,

    'tip_over_100_rows': payment0_tip_over_100,

    'payment0_total_rows': payment0_total_rows

}

{'tip_positive_rows': 1184792,
 'tip_over_100_rows': 2,
 'payment0_total_rows': 1368303}

In [ ]:
sum(payment0_tip_values), np.mean(payment0_tip_values), np.median(payment0_tip_values)

(182998970.44, np.float64(133.7415546410408), np.float64(3.05))

In [ ]:
top_tip_rows = []

tip_cols = [

    'tip_amount','payment_type','total_amount','fare_amount','tpep_pickup_datetime',

    'tpep_dropoff_datetime','VendorID','trip_distance'

]

for path in trip_files:

    df = pd.read_parquet(path, columns=[col for col in tip_cols if col in schema_columns]).copy()

    df['source_file'] = path.name

    df = df[df['payment_type'] == 0]

    if df.empty:

        continue

    top_tip_rows.append(df.nlargest(5, 'tip_amount'))



top_tip_df = pd.concat(top_tip_rows).nlargest(10, 'tip_amount') if top_tip_rows else pd.DataFrame()

payment0_cache['top_tip_rows'] = top_tip_df

top_tip_df

,tip_amount,payment_type,total_amount,fare_amount,tpep_pickup_datetime,tpep_dropoff_datetime,VendorID,trip_distance,source_file
3334938,1.333914e+08,0,-47.17,-1.333914e+08,2022-12-14 08:26:25,2022-12-14 08:43:25,2,0.00,yellow_tripdata_2022-12.parquet
3335231,4.446379e+07,0,21.08,-4.446377e+07,2022-12-14 09:24:56,2022-12-14 09:57:57,2,0.00,yellow_tripdata_2022-12.parquet
3160214,6.762000e+01,0,168.26,9.379000e+01,2022-07-28 01:35:00,2022-07-28 02:34:00,2,39.51,yellow_tripdata_2022-07.parquet
3436824,6.208000e+01,0,142.32,7.039000e+01,2022-06-03 15:15:35,2022-06-03 16:59:21,2,19.10,yellow_tripdata_2022-06.parquet
3077263,6.006000e+01,0,362.31,3.019500e+02,2022-08-06 12:52:00,2022-08-06 12:52:00,2,0.00,yellow_tripdata_2022-08.parquet
3067082,5.501000e+01,0,122.36,6.000000e+01,2022-08-03 15:25:45,2022-08-03 16:30:07,2,19.68,yellow_tripdata_2022-08.parquet
3573097,5.226000e+01,0,118.03,5.592000e+01,2022-05-27 05:11:01,2022-05-27 05:55:39,2,18.70,yellow_tripdata_2022-05.parquet
3130327,5.000000e+01,0,91.15,4.035000e+01,2022-09-17 15:41:13,2022-09-17 16:25:23,2,20.54,yellow_tripdata_2022-09.parquet
3125410,4.869000e+01,0,111.81,5.982000e+01,2022-09-16 15:57:00,2022-09-16 17:36:00,2,17.30,yellow_tripdata_2022-09.parquet
3153170,4.660000e+01,0,91.53,3.758000e+01,2022-07-26 07:36:00,2022-07-26 09:00:00,2,20.27,yellow_tripdata_2022-07.parquet
